# HydraY NNUE - HalfKA 8 king bucket sui dati COMPLETI (2,75B)

Runtime → Cambia tipo di runtime → **GPU (T4)**, poi Runtime → **Esegui tutte**.
Durata ~1h15. **Non lasciare la scheda inattiva.**

### Perché a tappe
Il runtime ha ~66 GB di disco locale contro un dataset da 88 GB, e il mount di
Drive tiene in cache locale tutto ciò che legge: un run in streaming riempie il
disco intorno al superbatch 28. Qui si addestra **una fetta alla volta**: ogni
tappa esce (liberando la cache), il file locale viene sovrascritto con la fetta
successiva e il training riprende dal checkpoint. La rete vede tutti i 2,75B
senza che nessuna fetta superi il disco.

L'ultimo test a 8 bucket girava su 1,18B e perdeva 12 Elo per **overfitting**
(training loss migliore, gioco peggiore). Questo run esiste per verificare se
con tutti i dati il verdetto cambia.

### Ogni cella si ferma se qualcosa non va
I comandi girano attraverso `sh()`, che solleva un'eccezione su exit code non
zero. Le righe `!` di Colab **non** lo fanno: è il motivo per cui un clone
mancante era passato inosservato per quattro tappe di fila.

In [ ]:
# --- helper: qualunque comando fallito ferma il notebook ---
import subprocess, os, sys, json

def sh(cmd):
    # L'output va letto e ristampato da Python: subprocess.run() senza capture
    # scrive sul file descriptor del KERNEL, che Colab non mostra nella cella.
    # Con la versione precedente ogni comando risultava muto e i numeri di loss
    # del training finivano nei log del runtime invece che sotto gli occhi.
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FALLITO (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')
print('\nGPU presente. Se la riga sopra non mostra una T4, cambia runtime.')

In [ ]:
# --- Drive + dataset ---
from google.colab import drive
drive.mount('/content/drive')

import glob
cand = glob.glob('/content/drive/MyDrive/**/hydray_v4_2754M_shuffled.bin', recursive=True)
assert cand, 'dataset non trovato su Drive'
DATA = cand[0]
SIZE = os.path.getsize(DATA)
assert SIZE == 88129087744, f'dimensione inattesa: {SIZE} (upload incompleto?)'
os.environ['DATA'] = DATA
print('dataset ok:', DATA)

# Geometria delle fette (MiB). 1000 MiB tenuti da parte come validation set:
# non entrano mai in addestramento, quindi la validation loss e' onesta.
SLICE_MIB = 20761
SKIPS     = [0, 20761, 41522, 62283]
TEST_SKIP, TEST_MIB = 83046, 1000
NET_ID    = 'hydray-h8-full'
TOTAL_SB  = 40
TRAINER   = '/content/thalfka8/nnue/trainer'
print('fette da', SLICE_MIB, 'MiB, test set da', TEST_MIB, 'MiB')

In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')

In [ ]:
# --- clone + verifica architettura ---
# La verifica non e' cerimoniale: un run precedente ha addestrato per un'ora
# la mappa a 4 bucket credendo di usare la 8, perche' il clone era sbagliato
# e il nome del checkpoint non dice nulla sul contenuto.
sh('rm -rf /content/thalfka8')
sh('git clone --depth 1 --branch halfka8 https://github.com/ThomasGhione/HydraY /content/thalfka8')

src = open(f'{TRAINER}/src/bin/sanity.rs').read()
assert 'const INPUT_BUCKETS: usize = 8;' in src, 'NON e il branch a 8 bucket'
tr = open(f'{TRAINER}/src/bin/trainer.rs').read()
assert 'start_superbatch,' in tr and 'load_from_checkpoint' in tr, 'branch privo del training a tappe'
print('branch halfka8 con 8 bucket e training a tappe: ok')

In [ ]:
# --- validation set (1000 MiB dalla coda, mai addestrato) ---
if not (os.path.exists('/content/test.bin') and os.path.getsize('/content/test.bin') == TEST_MIB*1024*1024):
    sh(f'dd if="$DATA" bs=1M skip={TEST_SKIP} count={TEST_MIB} of=/content/test.bin status=progress')
print('test.bin:', os.path.getsize('/content/test.bin'), 'byte')

In [ ]:
# --- una tappa: copia la fetta, poi addestra riprendendo dal checkpoint ---
def stage(n):
    skip     = SKIPS[n-1]
    start_sb = 1 + (n-1)*10
    if n == 1:
        resume = ''
    else:
        prev = f'checkpoints/{NET_ID}-{start_sb-1}'
        assert os.path.isdir(f'{TRAINER}/{prev}'), \
            f'checkpoint mancante: {prev} - la tappa precedente non ha finito'
        # <start_sb> <checkpoint>: l'indice e' ASSOLUTO, cosi' lo scheduler del
        # learning rate resta quello del run intero e il calo cade dove deve.
        resume = f'{start_sb} {prev}'
    print(f'=== TAPPA {n}: superbatch {start_sb}-{start_sb+9}, slice skip={skip} MiB ===')
    sh(f'dd if="$DATA" bs=1M skip={skip} count={SLICE_MIB} of=/content/data.bin status=progress')
    sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH CUDA_PATH=/usr/local/cuda '
       f'TEST_PATH=/content/test.bin STAGE_END={start_sb+9} '
       f'cargo run -r --bin trainer --features cuda -- '
       f'/content/data.bin {TOTAL_SB} {NET_ID} {resume}')
    sh('df -h /content | tail -1')
print('definita.')

In [ ]:
stage(1)   # superbatch 1-10   (~5 min copia + ~11 min con la compilazione)

In [ ]:
stage(2)   # superbatch 11-20  - deve stampare 'resuming from ... at superbatch 11'

In [ ]:
stage(3)   # superbatch 21-30

In [ ]:
stage(4)   # superbatch 31-40  - qui scende il learning rate

In [ ]:
# --- verifica finale e salvataggio su Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
assert sz == 6308928, f'taglia {sz}: NON e la rete a 8 bucket (la 4 pesa 3163200)'
print('quantised.bin:', sz, 'byte - mappa a 8 bucket confermata')
sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print('\nsalvato su Drive. Scarica quantised.bin e riporta:')
print(' - training loss finale')
print(' - VALIDATION loss finale')
print(' - i sanity eval qui sopra')

## Come leggere il risultato

La **validation loss** e' la novita': il test set non entra mai in
addestramento, quindi il confronto fra le due curve dice direttamente se la
rete generalizza.

- scendono insieme → i dati in piu' hanno fatto il loro lavoro
- la training scende e la validation no → **overfitting**, cioe' lo stesso
  difetto visto a 1,18B, e la mappa a 8 bucket va archiviata

Il giudice finale resta comunque l'SPRT **testa a testa** contro la rete a 4
bucket della 3.0.0 - non contro una baseline comune, perche' due misure
separate lascerebbero la differenza dentro il rumore.